In [ ]:
import sys
sys.path.append('../')

import torch
import random
import torch.backends.cudnn as cudnn
import numpy as np
from utils.dataset import GraphDataset_paired, GraphDataset_unpaired
from train.train_GCN import train_GCN_comp, train_GCN_sup
torch.manual_seed(0)
torch.cuda.manual_seed(0)
torch.cuda.manual_seed_all(0)
np.random.seed(0)
cudnn.benchmark = False
cudnn.deterministic = True
random.seed(0)

In [ ]:
device="cuda:0"

In [ ]:
data_dir="../data/data_angle/"
result_dir="../results/GCN_angle/"

In [ ]:
idx_list_test1=np.random.choice(list(range(1000)), 100, replace=False)
idx_list_train=[i for i in range(1000) if i not in idx_list_test1]
idx_list_train1_200=np.random.choice(idx_list_train,200, replace=False)
idx_list_train1_20=np.random.choice(idx_list_train1_200,20, replace=False)
idx_list_train2_180=[i for i in idx_list_train1_200 if i not in idx_list_train1_20]

In [ ]:
test1=GraphDataset_paired(idx_list_test1, data_dir, device)

In [ ]:
train1=GraphDataset_paired(idx_list_train1_200, data_dir, device)
train_GCN_sup(device, train1, test1, result_dir, ib_n=False, num_exp=1)
#sup: fully supervised baseline
#ib: inductive bias
#ib_n: node-level centering 
#note that message-level centering is not available for GCN.
#num_exp: number of experiments with the same seed

In [ ]:
train1=GraphDataset_paired(idx_list_train1_20, data_dir, device)
train2=GraphDataset_unpaired(idx_list_train2_180, data_dir, device)

train_GCN_comp(device, train1, train2, test1, result_dir, ib_n=True, num_exp=1)
#comp: complementary learning

In [ ]:
from utils.analysis import RMSE

print(RMSE(result_dir=result_dir, model="GCN", learning="sup", N_paired=200, N_total=200, ib="F", exp_list=[0]))
print(RMSE(result_dir=result_dir, model="GCN", learning="comp", N_paired=20, N_total=200, ib="T", exp_list=[0]))

#print (mean, std)